In [4]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

# Paramètres de connexion PostgreSQL
DB_USER = "postgres"
DB_PASSWORD = "zakaria"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "pfas_dw"

# Création de l'URL de connexion
DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Création du moteur SQLAlchemy
engine = create_engine(DATABASE_URL)

# Test de connexion
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.fetchone()[0])

PostgreSQL 18.2 on x86_64-windows, compiled by msvc-19.44.35222, 64-bit


In [5]:
from sqlalchemy import text

# Création des schémas du Data Warehouse
with engine.connect() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS dw;"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS mart;"))
    conn.commit()

print("Schémas dw et mart créés avec succès.")

Schémas dw et mart créés avec succès.


In [6]:
import pandas as pd
from pathlib import Path

# Dossier contenant les fichiers PROCESSED
PROCESSED_DIR = Path("../data/processed")

# Chargement des dimensions
dim_date = pd.read_parquet(PROCESSED_DIR / "dim_date.parquet")
dim_dataset = pd.read_parquet(PROCESSED_DIR / "dim_dataset.parquet")
dim_matrix = pd.read_parquet(PROCESSED_DIR / "dim_matrix.parquet")
dim_unit = pd.read_parquet(PROCESSED_DIR / "dim_unit.parquet")
dim_location = pd.read_parquet(PROCESSED_DIR / "dim_location.parquet")
dim_substance = pd.read_parquet(PROCESSED_DIR / "dim_substance.parquet")

# Chargement des tables de faits
fact_pfas_observation = pd.read_parquet(PROCESSED_DIR / "fact_pfas_observation.parquet")
fact_pfas_measurement = pd.read_parquet(PROCESSED_DIR / "fact_pfas_measurement.parquet")

print("Tables PROCESSED chargées avec succès.")

Tables PROCESSED chargées avec succès.


In [7]:
tables_summary = pd.DataFrame({
    "table": [
        "dim_date",
        "dim_dataset",
        "dim_matrix",
        "dim_unit",
        "dim_location",
        "dim_substance",
        "fact_pfas_observation",
        "fact_pfas_measurement"
    ],
    "rows": [
        dim_date.shape[0],
        dim_dataset.shape[0],
        dim_matrix.shape[0],
        dim_unit.shape[0],
        dim_location.shape[0],
        dim_substance.shape[0],
        fact_pfas_observation.shape[0],
        fact_pfas_measurement.shape[0]
    ],
    "columns": [
        dim_date.shape[1],
        dim_dataset.shape[1],
        dim_matrix.shape[1],
        dim_unit.shape[1],
        dim_location.shape[1],
        dim_substance.shape[1],
        fact_pfas_observation.shape[1],
        fact_pfas_measurement.shape[1]
    ]
})

display(tables_summary)

,table,rows,columns
0,dim_date,5738,8
1,dim_dataset,107,3
2,dim_matrix,14,3
3,dim_unit,4,3
4,dim_location,104123,8
5,dim_substance,295,6
6,fact_pfas_observation,929442,11
7,fact_pfas_measurement,21530916,14


In [8]:
import pandas as pd
import pyarrow.parquet as pq
import io
import csv
import gc
import time
from pathlib import Path
from sqlalchemy import text

In [9]:
# Dossier contenant les fichiers PROCESSED
PROCESSED_DIR = Path("../data/processed")

# Liste des fichiers à charger dans PostgreSQL
processed_files = {
    "dim_date": PROCESSED_DIR / "dim_date.parquet",
    "dim_dataset": PROCESSED_DIR / "dim_dataset.parquet",
    "dim_matrix": PROCESSED_DIR / "dim_matrix.parquet",
    "dim_unit": PROCESSED_DIR / "dim_unit.parquet",
    "dim_location": PROCESSED_DIR / "dim_location.parquet",
    "dim_substance": PROCESSED_DIR / "dim_substance.parquet",
    "fact_pfas_observation": PROCESSED_DIR / "fact_pfas_observation.parquet",
    "fact_pfas_measurement": PROCESSED_DIR / "fact_pfas_measurement.parquet"
}

# Vérification de l'existence des fichiers
for table_name, file_path in processed_files.items():
    print(table_name, "->", file_path.exists(), "|", file_path)

dim_date -> True | ..\data\processed\dim_date.parquet
dim_dataset -> True | ..\data\processed\dim_dataset.parquet
dim_matrix -> True | ..\data\processed\dim_matrix.parquet
dim_unit -> True | ..\data\processed\dim_unit.parquet
dim_location -> True | ..\data\processed\dim_location.parquet
dim_substance -> True | ..\data\processed\dim_substance.parquet
fact_pfas_observation -> True | ..\data\processed\fact_pfas_observation.parquet
fact_pfas_measurement -> True | ..\data\processed\fact_pfas_measurement.parquet


In [10]:
def get_parquet_row_count(parquet_path):
    # Lire uniquement les métadonnées du fichier Parquet
    # Cela évite de charger tout le fichier en mémoire
    parquet_file = pq.ParquetFile(parquet_path)
    return parquet_file.metadata.num_rows

In [11]:
def quote_identifier(name):
    # Protège les noms de colonnes ou de tables pour PostgreSQL
    return '"' + str(name).replace('"', '""') + '"'


def copy_dataframe_to_postgres(df, table_name, schema="dw"):
    # Copie un DataFrame vers PostgreSQL avec la commande COPY
    buffer = io.StringIO()

    # Conversion du DataFrame en CSV en mémoire
    df.to_csv(
        buffer,
        index=False,
        header=False,
        sep="\t",
        na_rep="\\N",
        quoting=csv.QUOTE_MINIMAL
    )

    buffer.seek(0)

    columns = ", ".join([quote_identifier(col) for col in df.columns])

    copy_sql = f"""
        COPY {quote_identifier(schema)}.{quote_identifier(table_name)} ({columns})
        FROM STDIN
        WITH (
            FORMAT CSV,
            DELIMITER E'\\t',
            NULL '\\N',
            HEADER FALSE
        );
    """

    raw_conn = engine.raw_connection()

    try:
        cursor = raw_conn.cursor()
        cursor.copy_expert(copy_sql, buffer)
        raw_conn.commit()
    except Exception as e:
        raw_conn.rollback()
        raise e
    finally:
        cursor.close()
        raw_conn.close()

In [13]:
def load_parquet_to_postgres(parquet_path, table_name, schema="dw", batch_size=100_000):
    # Chargement d'un fichier Parquet vers PostgreSQL par lots
    parquet_file = pq.ParquetFile(parquet_path)

    total_rows = parquet_file.metadata.num_rows
    loaded_rows = 0
    first_batch = True

    start_time = time.time()

    print(f"Chargement de {table_name}")
    print(f"Nombre total de lignes à charger : {total_rows}")

    for batch_number, batch in enumerate(parquet_file.iter_batches(batch_size=batch_size), start=1):
        # Convertir le batch Arrow en DataFrame Pandas
        chunk = batch.to_pandas()

        # Créer la table vide au premier batch
        if first_batch:
            chunk.head(0).to_sql(
                table_name,
                engine,
                schema=schema,
                if_exists="replace",
                index=False
            )
            first_batch = False

        # Charger le batch dans PostgreSQL
        copy_dataframe_to_postgres(chunk, table_name, schema=schema)

        loaded_rows += len(chunk)

        print(
            f"Batch {batch_number} chargé | "
            f"{loaded_rows}/{total_rows} lignes"
        )

        # Nettoyage mémoire
        del chunk
        gc.collect()

    elapsed_time = round(time.time() - start_time, 2)

    print(f"Chargement terminé pour {table_name}")
    print(f"Lignes chargées : {loaded_rows}")
    print(f"Temps total : {elapsed_time} secondes")

### 4.1 Chargement des dimensions

Les dimensions sont chargées en premier, car elles servent de référence aux tables de faits.

In [14]:
dimension_tables = [
    "dim_date",
    "dim_dataset",
    "dim_matrix",
    "dim_unit",
    "dim_location",
    "dim_substance"
]

for table_name in dimension_tables:
    load_parquet_to_postgres(
        parquet_path=processed_files[table_name],
        table_name=table_name,
        schema="dw",
        batch_size=100_000
    )

Chargement de dim_date
Nombre total de lignes à charger : 5738
Batch 1 chargé | 5738/5738 lignes
Chargement terminé pour dim_date
Lignes chargées : 5738
Temps total : 0.8 secondes
Chargement de dim_dataset
Nombre total de lignes à charger : 107
Batch 1 chargé | 107/107 lignes
Chargement terminé pour dim_dataset
Lignes chargées : 107
Temps total : 0.12 secondes
Chargement de dim_matrix
Nombre total de lignes à charger : 14
Batch 1 chargé | 14/14 lignes
Chargement terminé pour dim_matrix
Lignes chargées : 14
Temps total : 0.12 secondes
Chargement de dim_unit
Nombre total de lignes à charger : 4
Batch 1 chargé | 4/4 lignes
Chargement terminé pour dim_unit
Lignes chargées : 4
Temps total : 0.1 secondes
Chargement de dim_location
Nombre total de lignes à charger : 104123
Batch 1 chargé | 100000/104123 lignes
Batch 2 chargé | 104123/104123 lignes
Chargement terminé pour dim_location
Lignes chargées : 104123
Temps total : 2.0 secondes
Chargement de dim_substance
Nombre total de lignes à charg

In [15]:
def get_postgres_row_count(table_name, schema="dw"):
    # Compter le nombre de lignes dans une table PostgreSQL
    query = text(f'SELECT COUNT(*) FROM "{schema}"."{table_name}";')

    with engine.connect() as conn:
        count = conn.execute(query).scalar()

    return count


dimension_check = []

for table_name in dimension_tables:
    local_count = get_parquet_row_count(processed_files[table_name])
    postgres_count = get_postgres_row_count(table_name, schema="dw")

    dimension_check.append({
        "table": table_name,
        "local_rows": local_count,
        "postgres_rows": postgres_count,
        "status": "OK" if local_count == postgres_count else "CHECK"
    })

dimension_check_df = pd.DataFrame(dimension_check)

display(dimension_check_df)

,table,local_rows,postgres_rows,status
0,dim_date,5738,5738,OK
1,dim_dataset,107,107,OK
2,dim_matrix,14,14,OK
3,dim_unit,4,4,OK
4,dim_location,104123,104123,OK
5,dim_substance,295,295,OK


### 4.2 Chargement de `fact_pfas_observation`

La table `fact_pfas_observation` représente le niveau global des observations.

Son grain est :

Une ligne = une observation environnementale globale.

Cette table contient `pfas_sum` et les indicateurs qualité associés.

In [16]:
load_parquet_to_postgres(
    parquet_path=processed_files["fact_pfas_observation"],
    table_name="fact_pfas_observation",
    schema="dw",
    batch_size=100_000
)

Chargement de fact_pfas_observation
Nombre total de lignes à charger : 929442
Batch 1 chargé | 100000/929442 lignes
Batch 2 chargé | 200000/929442 lignes
Batch 3 chargé | 300000/929442 lignes
Batch 4 chargé | 400000/929442 lignes
Batch 5 chargé | 500000/929442 lignes
Batch 6 chargé | 600000/929442 lignes
Batch 7 chargé | 700000/929442 lignes
Batch 8 chargé | 800000/929442 lignes
Batch 9 chargé | 900000/929442 lignes
Batch 10 chargé | 929442/929442 lignes
Chargement terminé pour fact_pfas_observation
Lignes chargées : 929442
Temps total : 14.2 secondes


In [17]:
local_count = get_parquet_row_count(processed_files["fact_pfas_observation"])
postgres_count = get_postgres_row_count("fact_pfas_observation", schema="dw")

print("Local :", local_count)
print("PostgreSQL :", postgres_count)
print("Statut :", "OK" if local_count == postgres_count else "CHECK")

Local : 929442
PostgreSQL : 929442
Statut : OK


In [ ]:
### 4.3 Chargement de `fact_pfas_measurement`

La table `fact_pfas_measurement` représente le niveau détaillé des mesures PFAS.

Son grain est :

Une ligne = une substance PFAS mesurée dans une observation.

Cette table est volumineuse, car elle contient plus de 21 millions de lignes.  
Elle est donc chargée progressivement par lots.

In [19]:
load_parquet_to_postgres(
    parquet_path=processed_files["fact_pfas_measurement"],
    table_name="fact_pfas_measurement",
    schema="dw",
    batch_size=100_000
)

Chargement de fact_pfas_measurement
Nombre total de lignes à charger : 21530916
Batch 1 chargé | 100000/21530916 lignes
Batch 2 chargé | 200000/21530916 lignes
Batch 3 chargé | 300000/21530916 lignes
Batch 4 chargé | 400000/21530916 lignes
Batch 5 chargé | 500000/21530916 lignes
Batch 6 chargé | 600000/21530916 lignes
Batch 7 chargé | 700000/21530916 lignes
Batch 8 chargé | 800000/21530916 lignes
Batch 9 chargé | 900000/21530916 lignes
Batch 10 chargé | 1000000/21530916 lignes
Batch 11 chargé | 1100000/21530916 lignes
Batch 12 chargé | 1200000/21530916 lignes
Batch 13 chargé | 1300000/21530916 lignes
Batch 14 chargé | 1400000/21530916 lignes
Batch 15 chargé | 1500000/21530916 lignes
Batch 16 chargé | 1600000/21530916 lignes
Batch 17 chargé | 1700000/21530916 lignes
Batch 18 chargé | 1800000/21530916 lignes
Batch 19 chargé | 1900000/21530916 lignes
Batch 20 chargé | 2000000/21530916 lignes
Batch 21 chargé | 2100000/21530916 lignes
Batch 22 chargé | 2200000/21530916 lignes
Batch 23 charg

In [20]:
local_count = get_parquet_row_count(processed_files["fact_pfas_measurement"])
postgres_count = get_postgres_row_count("fact_pfas_measurement", schema="dw")

print("Local :", local_count)
print("PostgreSQL :", postgres_count)
print("Statut :", "OK" if local_count == postgres_count else "CHECK")

Local : 21530916
PostgreSQL : 21530916
Statut : OK


###on crée ensuite Des index  sur les principales clés des tables de faits afin d’accélérer les futures requêtes SQL , éviter de scaner completement la les tableet l’utilisation dans Metabase(amélioration des dashbords) 

In [21]:
index_queries = [
    # Index sur fact_pfas_observation
    'CREATE INDEX IF NOT EXISTS idx_fact_obs_date_id ON dw.fact_pfas_observation(date_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_obs_location_id ON dw.fact_pfas_observation(location_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_obs_dataset_id ON dw.fact_pfas_observation(dataset_dw_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_obs_matrix_id ON dw.fact_pfas_observation(matrix_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_obs_unit_id ON dw.fact_pfas_observation(unit_id);',

    # Index sur fact_pfas_measurement
    'CREATE INDEX IF NOT EXISTS idx_fact_meas_observation_id ON dw.fact_pfas_measurement(observation_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_meas_date_id ON dw.fact_pfas_measurement(date_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_meas_location_id ON dw.fact_pfas_measurement(location_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_meas_dataset_id ON dw.fact_pfas_measurement(dataset_dw_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_meas_matrix_id ON dw.fact_pfas_measurement(matrix_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_meas_substance_id ON dw.fact_pfas_measurement(substance_id);'
]

with engine.connect() as conn:
    for query in index_queries:
        conn.execute(text(query))
    conn.commit()

print("Index créés avec succès.")

Index créés avec succès.


In [ ]:
## 6. Création des vues analytiques dans le schéma `mart`

Les vues du schéma `mart` servent à préparer les indicateurs pour Metabase.

Elles permettent de simplifier les requêtes et d’éviter de reconstruire les mêmes jointures dans chaque graphique.

Nous allons créer plusieurs vues :
- vue générale des KPI ;
- analyse par pays ;
- analyse par matrice ;
- analyse par année ;
- analyse des substances ;
- analyse des valeurs extrêmes.

In [22]:
# Vue générale des KPI du projet
query_overview_kpis = """
CREATE OR REPLACE VIEW mart.vw_overview_kpis AS
SELECT
    (SELECT COUNT(*) FROM dw.fact_pfas_observation) AS total_observations,
    (SELECT COUNT(*) FROM dw.fact_pfas_measurement) AS total_measurements,
    (SELECT COUNT(*) FROM dw.dim_location) AS total_locations,
    (SELECT COUNT(*) FROM dw.dim_substance) AS total_substances,
    (SELECT COUNT(*) FROM dw.dim_dataset) AS total_datasets,
    (SELECT ROUND(AVG(CASE WHEN is_positive_pfas_sum THEN 1 ELSE 0 END) * 100, 2)
     FROM dw.fact_pfas_observation) AS positive_observation_rate_percent,
    (SELECT ROUND(AVG(CASE WHEN is_zero_pfas_sum THEN 1 ELSE 0 END) * 100, 2)
     FROM dw.fact_pfas_observation) AS zero_observation_rate_percent,
    (SELECT MAX(pfas_sum)
     FROM dw.fact_pfas_observation) AS max_pfas_sum;
"""

with engine.connect() as conn:
    conn.execute(text(query_overview_kpis))
    conn.commit()

print("Vue mart.vw_overview_kpis créée.")

Vue mart.vw_overview_kpis créée.


In [28]:
query_country = """
CREATE OR REPLACE VIEW mart.vw_pfas_by_country AS
SELECT
    l.country,
    COUNT(*) AS observation_count,
    SUM(CASE WHEN f.is_positive_pfas_sum THEN 1 ELSE 0 END) AS positive_observation_count,
    ROUND(AVG(CASE WHEN f.is_positive_pfas_sum THEN 1 ELSE 0 END) * 100, 2) AS positive_rate_percent,
    AVG(f.pfas_sum) AS avg_pfas_sum,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY f.pfas_sum) AS median_pfas_sum,
    MAX(f.pfas_sum) AS max_pfas_sum
FROM dw.fact_pfas_observation f
LEFT JOIN dw.dim_location l
    ON f.location_id = l.location_id
GROUP BY l.country
ORDER BY observation_count DESC;
"""

with engine.connect() as conn:
    conn.execute(text(query_country))
    conn.commit()

print("Vue mart.vw_pfas_by_country créée.")

Vue mart.vw_pfas_by_country créée.


In [29]:
query_matrix = """
CREATE OR REPLACE VIEW mart.vw_pfas_by_matrix AS
SELECT
    m.matrix_standardized,
    u.unit_standardized,
    COUNT(*) AS observation_count,
    SUM(CASE WHEN f.is_positive_pfas_sum THEN 1 ELSE 0 END) AS positive_observation_count,
    ROUND(AVG(CASE WHEN f.is_positive_pfas_sum THEN 1 ELSE 0 END) * 100, 2) AS positive_rate_percent,
    AVG(f.pfas_sum) AS avg_pfas_sum,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY f.pfas_sum) AS median_pfas_sum,
    MAX(f.pfas_sum) AS max_pfas_sum
FROM dw.fact_pfas_observation f
LEFT JOIN dw.dim_matrix m
    ON f.matrix_id = m.matrix_id
LEFT JOIN dw.dim_unit u
    ON f.unit_id = u.unit_id
GROUP BY m.matrix_standardized, u.unit_standardized
ORDER BY observation_count DESC;
"""

with engine.connect() as conn:
    conn.execute(text(query_matrix))
    conn.commit()

print("Vue mart.vw_pfas_by_matrix créée.")

Vue mart.vw_pfas_by_matrix créée.


In [30]:
query_year = """
CREATE OR REPLACE VIEW mart.vw_pfas_by_year AS
SELECT
    d.year_clean,
    COUNT(*) AS observation_count,
    SUM(CASE WHEN f.is_positive_pfas_sum THEN 1 ELSE 0 END) AS positive_observation_count,
    ROUND(AVG(CASE WHEN f.is_positive_pfas_sum THEN 1 ELSE 0 END) * 100, 2) AS positive_rate_percent,
    AVG(f.pfas_sum) AS avg_pfas_sum,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY f.pfas_sum) AS median_pfas_sum,
    MAX(f.pfas_sum) AS max_pfas_sum
FROM dw.fact_pfas_observation f
LEFT JOIN dw.dim_date d
    ON f.date_id = d.date_id
GROUP BY d.year_clean
ORDER BY d.year_clean;
"""

with engine.connect() as conn:
    conn.execute(text(query_year))
    conn.commit()

print("Vue mart.vw_pfas_by_year créée.")

Vue mart.vw_pfas_by_year créée.


In [31]:
query_substances = """
CREATE OR REPLACE VIEW mart.vw_top_substances AS
SELECT
    s.substance_name,
    s.substance_standardized,
    s.cas_id,
    s.isomer,
    COUNT(*) AS measurement_count,
    SUM(CASE WHEN f.is_positive_pfas_value THEN 1 ELSE 0 END) AS positive_measurement_count,
    ROUND(AVG(CASE WHEN f.is_positive_pfas_value THEN 1 ELSE 0 END) * 100, 2) AS positive_rate_percent,
    AVG(f.pfas_value) AS avg_pfas_value,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY f.pfas_value) AS median_pfas_value,
    MAX(f.pfas_value) AS max_pfas_value
FROM dw.fact_pfas_measurement f
LEFT JOIN dw.dim_substance s
    ON f.substance_id = s.substance_id
GROUP BY
    s.substance_name,
    s.substance_standardized,
    s.cas_id,
    s.isomer
ORDER BY measurement_count DESC;
"""

with engine.connect() as conn:
    conn.execute(text(query_substances))
    conn.commit()

print("Vue mart.vw_top_substances créée.")

Vue mart.vw_top_substances créée.


In [32]:
query_extremes = """
CREATE OR REPLACE VIEW mart.vw_extreme_observations AS
WITH q99_by_group AS (
    SELECT
        matrix_id,
        unit_id,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY pfas_sum) AS q99_pfas_sum
    FROM dw.fact_pfas_observation
    WHERE pfas_sum > 0
    GROUP BY matrix_id, unit_id
),

obs_with_extreme AS (
    SELECT
        f.*,
        q.q99_pfas_sum
    FROM dw.fact_pfas_observation f
    LEFT JOIN q99_by_group q
        ON f.matrix_id = q.matrix_id
       AND f.unit_id = q.unit_id
)

SELECT
    f.fact_observation_id,
    f.observation_id,
    f.pfas_sum,
    f.q99_pfas_sum,
    d.year_clean,
    l.country,
    l.city,
    l.site_name,
    l.lat,
    l.lon,
    m.matrix_standardized,
    u.unit_standardized,
    ds.dataset_name
FROM obs_with_extreme f
LEFT JOIN dw.dim_date d
    ON f.date_id = d.date_id
LEFT JOIN dw.dim_location l
    ON f.location_id = l.location_id
LEFT JOIN dw.dim_matrix m
    ON f.matrix_id = m.matrix_id
LEFT JOIN dw.dim_unit u
    ON f.unit_id = u.unit_id
LEFT JOIN dw.dim_dataset ds
    ON f.dataset_dw_id = ds.dataset_dw_id
WHERE f.pfas_sum > f.q99_pfas_sum
ORDER BY f.pfas_sum DESC;
"""

with engine.connect() as conn:
    conn.execute(text(query_extremes))
    conn.commit()

print("Vue mart.vw_extreme_observations créée.")

Vue mart.vw_extreme_observations créée.


In [33]:
overview_kpis = pd.read_sql(
    "SELECT * FROM mart.vw_overview_kpis;",
    engine
)

display(overview_kpis)

,total_observations,total_measurements,total_locations,total_substances,total_datasets,positive_observation_rate_percent,zero_observation_rate_percent,max_pfas_sum
0,929442,21530916,104123,295,107,38.87,61.13,800245054.0


In [34]:
country_view = pd.read_sql(
    "SELECT * FROM mart.vw_pfas_by_country LIMIT 10;",
    engine
)

display(country_view)

,country,observation_count,positive_observation_count,positive_rate_percent,avg_pfas_sum,median_pfas_sum,max_pfas_sum
0,France,669993,194406,29.02,2418.714480,0.0000,509000000.0
1,United Kingdom,62224,61187,98.33,6612.571942,8.4800,106550000.0
2,Belgium,57187,39011,68.22,232334.748234,89.2000,800245054.0
3,Italy,50869,31901,62.71,957.025964,3.7400,7132440.0
4,Austria,41000,3605,8.79,903.195681,0.0000,733800.0
5,Germany,25272,18032,71.35,562.181699,6.0000,484490.0
6,Denmark,10237,5477,53.50,734.407675,1.3000,355080.0
7,Netherlands,6447,3523,54.65,32540.142943,240.0000,80000000.0
8,Lithuania,1583,616,38.91,51.377469,0.0000,5280.0
9,Unknown,1417,987,69.65,38.157999,0.2805,3300.0
